# **Marketing Campaign AB Testing EDA.**

## **Load Data.**

In [1]:
import os
import pandas as pd

DATASETS_PATH = os.getenv("DATASETS_PATH")
DATASETS_PATH = DATASETS_PATH + "/MarketingA_B_Testing/"

data_df = pd.read_csv(DATASETS_PATH + "marketing_AB.csv")

print("Data: ")
print(data_df)


Data: 
        Unnamed: 0  user id test group  converted  total ads most ads day  \
0                0  1069124         ad      False        130       Monday   
1                1  1119715         ad      False         93      Tuesday   
2                2  1144181         ad      False         21      Tuesday   
3                3  1435133         ad      False        355      Tuesday   
4                4  1015700         ad      False        276       Friday   
...            ...      ...        ...        ...        ...          ...   
588096      588096  1278437         ad      False          1      Tuesday   
588097      588097  1327975         ad      False          1      Tuesday   
588098      588098  1038442         ad      False          3      Tuesday   
588099      588099  1496395         ad      False          1      Tuesday   
588100      588100  1237779         ad      False          1      Tuesday   

        most ads hour  
0                  20  
1                  2

## **Calculate how many groups are there for testing and how many users each group has.**

In [2]:
print(data_df["test group"].value_counts())

test group
ad     564577
psa     23524
Name: count, dtype: int64


### There are two groups in this dataset. The target group called "ad" and control one - "psa". The numbers of users in both groups are very imbalanced, but as long as they are chosen randomly it is not a problem for statistical hypothesis testing.

## **Calculate number of adds conversions comparing to the overall number of experiments.**

In [3]:
print(data_df["converted"].value_counts())

converted
False    573258
True      14843
Name: count, dtype: int64


In [4]:
print("The overall conversion rate:")
print(data_df["converted"].value_counts(normalize=True))

The overall conversion rate:
converted
False    0.974761
True     0.025239
Name: proportion, dtype: float64


In [5]:
data_df.groupby("test group")["converted"].value_counts(normalize=True)

test group  converted
ad          False        0.974453
            True         0.025547
psa         False        0.982146
            True         0.017854
Name: proportion, dtype: float64

### The dataset exhibits a strong class imbalance toward non-conversion, which is expected in real-world marketing settings. Additionally, conversion rates differ slightly between treatment groups. However, neither the class imbalance nor the group size differences pose a limitation for subsequent statistical testing, as standard methods such as proportion-based hypothesis tests and bootstrap procedures are robust to these conditions given the large sample size.

## **Calculate the descriptive statistics of the total number of adds shown to users.**

In [6]:
print(data_df.groupby("test group")["total ads"].describe())

               count       mean        std  min  25%   50%   75%     max
test group                                                              
ad          564577.0  24.823365  43.750456  1.0  4.0  13.0  27.0  2065.0
psa          23524.0  24.761138  42.860720  1.0  4.0  12.0  26.0   907.0


### The levels of user exposure in both groups are comparable, so this information can be further used in ordert ot test hypothesises about dependency of convertion rate on adds exposure level.

## **Calculate the descriptive statistics for the days with most adds.**

In [7]:
print(data_df.groupby("test group")["most ads day"].describe())

             count unique       top   freq
test group                                
ad          564577      7    Friday  88805
psa          23524      7  Thursday   3905


In [8]:
data_df.groupby("most ads day")["converted"].mean()

most ads day
Friday       0.022212
Monday       0.032812
Saturday     0.021051
Sunday       0.024476
Thursday     0.021571
Tuesday      0.029840
Wednesday    0.024942
Name: converted, dtype: float64

In [9]:
data_df.groupby("most ads day")["converted"].count()

most ads day
Friday       92608
Monday       87073
Saturday     81660
Sunday       85391
Thursday     82982
Tuesday      77479
Wednesday    80908
Name: converted, dtype: int64

### Conversion counts are relatively evenly distributed across weekdays, indicating no obvious structural imbalance. This supports the validity of proceeding with formal statistical tests to assess any potential weekday effect on conversion behavior.

## **Calculate the descriptive statistics for the hours with most adds.**

In [10]:
data_df["most ads hour"] = data_df["most ads hour"].astype("category")
print(data_df.groupby("test group")["most ads hour"].describe())

             count  unique  top   freq
test group                            
ad          564577      24   13  45485
psa          23524      24   13   2170


In [11]:
data_df.groupby("most ads hour")["converted"].mean()

most ads hour
0     0.018425
1     0.012911
2     0.007313
3     0.010452
4     0.015235
5     0.020915
6     0.022244
7     0.018111
8     0.019516
9     0.019191
10    0.021521
11    0.022116
12    0.023828
13    0.024677
14    0.028063
15    0.029653
16    0.030772
17    0.028210
18    0.027380
19    0.026720
20    0.029803
21    0.028923
22    0.026105
23    0.022662
Name: converted, dtype: float64

In [12]:
data_df.groupby("most ads hour")["converted"].count()

most ads hour
0      5536
1      4802
2      5333
3      2679
4       722
5       765
6      2068
7      6405
8     17627
9     31004
10    38939
11    46210
12    47298
13    47655
14    45648
15    44683
16    37567
17    34988
18    32323
19    30352
20    28923
21    29976
22    26432
23    20166
Name: converted, dtype: int64

### The distribution of user activity varies significantly across hours, with expected lower engagement during nighttime periods. However, conversion rates remain relatively stable across hours, with only mild variation. This suggests that temporal activity patterns influence volume but do not strongly distort conversion behavior at an aggregate level. No evidence of data inconsistency or corruption is observed based on these distributions.

## **Conclusions:**
### The exploratory data analysis shows that dataset is well structured and suitable for statistical inference. Target variable is highly imbalanced, with low overall conversion rate, which is expected in marketing response data.

### Conversion behavior shows small variation across temporal features (hour and weekday) and slightly higher conversion rate in treatment (ad) group compared to control (psa) group. However, these observations are only descriptive and do not consider sampling variability or possible confounding factors.

### Importantly, no structural anomalies, inconsistencies, or signs of data corruption were found. Group sizes are large enough, and both temporal and treatment groups are well represented.

### Therefore, dataset is appropriate for formal hypothesis testing to check whether observed differences in conversion rates across treatment and temporal dimensions are statistically significant or caused by random variation.